In [1]:
# ==========================================
# IMPORTS
# ==========================================
#import os, json, torch, numpy as np, time
#import torchvision
#from torch.utils.data import Dataset, DataLoader
#from PIL import Image, ImageDraw
#import torchvision.transforms.functional as F
#from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
#from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
#import torch.optim as optim
#from tqdm import tqdm
#import warnings
#warnings.filterwarnings('ignore')

# ==========================================
# DEVICE SETUP
# ==========================================
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
##print(f"🚀 Running on: {device}")

# ==========================================
# PATHS
# ==========================================
#BASE = "/kaggle/input/datasets/kunalnarang47/deepfashion-redwing/deepfashion2_pruned"
#TRAIN_IMG = BASE + "/train/image"
#TRAIN_ANNO = BASE + "/train/annos"

# ==========================================
# CATEGORY MAP (TOP 5)
# ==========================================
#TOP5 = [1,8,7,2,9]
#LABEL_MAP = {cid: i+1 for i,cid in enumerate(TOP5)}

#print("✅ Category mapping ready")


# ==========================================
# DATASET CLASS
# ==========================================
##class DeepFashionDataset(Dataset):
 #   def __init__(self, img_dir, anno_dir, limit=10000):
  #      self.img_dir = img_dir
 #       self.anno_dir = anno_dir

        # 🔥 LIMIT DATASET FOR SPEED
   #     self.files = os.listdir(anno_dir)[:limit]
#
   #     print(f"📦 Loaded {len(self.files)} annotation files")

  #  def __len__(self):
    #    return len(self.files)

  #  def __getitem__(self, idx):
  #      file = self.files[idx]

  #      img_path = os.path.join(self.img_dir, file.replace(".json",".jpg"))
  #      json_path = os.path.join(self.anno_dir, file)

        # Skip if image missing
  #      if not os.path.exists(img_path):
   #         return None

     #   img = Image.open(img_path).convert("RGB")
     #   w, h = img.size
#
        # 🔥 RESIZE FOR SPEED
    #    img = img.resize((512,512))

        # ===== LOAD JSON (FIXED BUG HERE) =====
    #    with open(json_path) as f:
    #        data = json.load(f)

        # Safety check
    #    if not isinstance(data, dict):
    #        return None

     #   boxes, labels, masks = [], [], []

        # ===== LOOP THROUGH OBJECTS =====
   #     for key in data:
    #        item = data[key]

            # Skip bad entries
    #        if "category_id" not in item:
     #           continue

       #     cid = item["category_id"]

            # Only keep top 5 classes
  #          if cid not in LABEL_MAP:
    #            continue

     #       segs = item.get("segmentation", [])

      #      mask = Image.new("L", (w,h), 0)
   #         draw = ImageDraw.Draw(mask)

    #        for seg in segs:
  #             if len(seg) < 6:
    #                continue
    #            pts = np.array(seg).reshape(-1,2)
    ##            draw.polygon([tuple(p) for p in pts], fill=1)
#
            # Resize mask same as image
    #        mask = mask.resize((512,512))
     #       mask_np = np.array(mask)

    #        if mask_np.sum() == 0:
    #            continue

            # Bounding box from mask
   #         pos = np.where(mask_np)
  #          xmin, xmax = np.min(pos[1]), np.max(pos[1])
  #          ymin, ymax = np.min(pos[0]), np.max(pos[0])
##
  #          boxes.append([xmin,ymin,xmax,ymax])
  #          labels.append(LABEL_MAP[cid])
  #          masks.append(mask_np)

 #       if len(boxes) == 0:
 #           return None

 #       target = {
 ##           "boxes": torch.tensor(boxes,dtype=torch.float32),
 #           "labels": torch.tensor(labels,dtype=torch.int64),
 #           "masks": torch.tensor(np.array(masks),dtype=torch.uint8),
 #           "image_id": torch.tensor([idx]),
#            "area": torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes]),
#            "iscrowd": torch.zeros((len(labels),),dtype=torch.int64)
 #       }

#        return F.to_tensor(img), target


# ==========================================
# COLLATE FUNCTION
# ==========================================
#def collate_fn(batch):
#    batch = [b for b in batch if b is not None]
#    if not batch:
#        return ()
#    return tuple(zip(*batch))


# ==========================================
# DATALOADER
# ==========================================
#train_loader = DataLoader(
#    DeepFashionDataset(TRAIN_IMG, TRAIN_ANNO),
#    batch_size=4,
#    shuffle=True,
#    num_workers=2,
#    pin_memory=True,
#    collate_fn=collate_fn
#)

#print("✅ DataLoader ready")


# ==========================================
# MODEL SETUP
# ==========================================
#print("🧠 Loading Mask R-CNN...")

#model = torchvision.models.detection.maskrcnn_resnet50_fpn(
#    weights='DEFAULT',
#    min_size=512,
#    max_size=512
#)

#NUM_CLASSES = 6

# Replace heads
#in_features = model.roi_heads.box_predictor.cls_score.in_features
#model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
#
#in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
#model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, NUM_CLASSES)

#model.to(device)
#
#print("✅ Model ready")


# ==========================================
# TRAINING SETUP
# ==========================================
#optimizer = optim.AdamW(model.parameters(), lr=1e-5)
#scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
#scaler = torch.cuda.amp.GradScaler()

#num_epochs = 10

#print("\n🚀 Training Started")


# ==========================================
# TRAIN LOOP
# ==========================================
#for epoch in range(num_epochs):
#    start_time = time.time()
#
#    model.train()
#    total_loss = 0

    # tqdm progress bar
 #   train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

#    for batch in train_bar:
#        if not batch:
#            continue

#        images, targets = batch
#        images = [img.to(device) for img in images]
#        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

#        optimizer.zero_grad()

        # Mixed precision (faster)
#        with torch.cuda.amp.autocast():
#            loss_dict = model(images, targets)
#            loss = sum(loss_dict.values())

#        scaler.scale(loss).backward()
#        scaler.step(optimizer)
 #       scaler.update()

 #       total_loss += loss.item()

        # Show live loss
#        train_bar.set_postfix(loss=f"{loss.item():.4f}")

#    scheduler.step()

#    end_time = time.time()

#    print(f"✅ Epoch {epoch+1} completed")
#    print(f"   Loss: {total_loss:.2f}")
#    print(f"   Time: {(end_time-start_time)/60:.2f} minutes\n")


# ==========================================
# SAVE MODEL
# ==========================================
#torch.save(model.state_dict(), "/kaggle/working/maskrcnn_final.pth")

#print("🎉 TRAINING COMPLETE — MODEL SAVED")

In [2]:
# ==========================================
# SCRATCH MASK R-CNN (FINAL SAFE VERSION)
# ==========================================
import os, json, torch, numpy as np, time
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import torchvision.transforms.functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torch.optim as optim
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Scratch Training on: {device}")

# ==========================================
# PATHS
# ==========================================
BASE = "/kaggle/input/datasets/kunalnarang47/deepfashion-redwing/deepfashion2_pruned"
TRAIN_IMG = BASE + "/train/image"
TRAIN_ANNO = BASE + "/train/annos"

# ==========================================
# CATEGORY MAP
# ==========================================
TOP5 = [1,8,7,2,9]
LABEL_MAP = {cid: i+1 for i,cid in enumerate(TOP5)}

# ==========================================
# DATASET
# ==========================================
class DeepFashionDataset(Dataset):
    def __init__(self, img_dir, anno_dir, limit=10000):
        self.img_dir = img_dir
        self.anno_dir = anno_dir
        self.files = os.listdir(anno_dir)[:limit]

        print(f"📦 Scratch dataset size: {len(self.files)}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        img_path = os.path.join(self.img_dir, file.replace(".json",".jpg"))
        json_path = os.path.join(self.anno_dir, file)

        if not os.path.exists(img_path):
            return None

        img = Image.open(img_path).convert("RGB")
        w,h = img.size
        img = img.resize((512,512))

        with open(json_path) as f:
            data = json.load(f)

        if not isinstance(data, dict):
            return None

        boxes, labels, masks = [], [], []

        for key in data:
            item = data[key]

            if "category_id" not in item:
                continue

            cid = item["category_id"]
            if cid not in LABEL_MAP:
                continue

            segs = item.get("segmentation", [])

            mask = Image.new("L", (w,h), 0)
            draw = ImageDraw.Draw(mask)

            for seg in segs:
                if len(seg) < 6: continue
                pts = np.array(seg).reshape(-1,2)
                draw.polygon([tuple(p) for p in pts], fill=1)

            mask = mask.resize((512,512))
            mask_np = np.array(mask)

            if mask_np.sum() == 0:
                continue

            pos = np.where(mask_np)
            xmin,xmax = np.min(pos[1]), np.max(pos[1])
            ymin,ymax = np.min(pos[0]), np.max(pos[0])

            boxes.append([xmin,ymin,xmax,ymax])
            labels.append(LABEL_MAP[cid])
            masks.append(mask_np)

        if len(boxes)==0:
            return None

        target = {
            "boxes": torch.tensor(boxes,dtype=torch.float32),
            "labels": torch.tensor(labels,dtype=torch.int64),
            "masks": torch.tensor(np.array(masks),dtype=torch.uint8)
        }

        return F.to_tensor(img), target


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return ()
    return tuple(zip(*batch))


train_loader = DataLoader(
    DeepFashionDataset(TRAIN_IMG, TRAIN_ANNO),
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

# ==========================================
# MODEL (SCRATCH)
# ==========================================
print("🧠 Initializing SCRATCH model...")

model = torchvision.models.detection.maskrcnn_resnet50_fpn(
    weights=None,   # 🔥 SCRATCH
    min_size=512,
    max_size=512
)

NUM_CLASSES = 6

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, NUM_CLASSES)

model.to(device)

# ==========================================
# TRAIN
# ==========================================
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 10

print("\n🚀 Scratch Training Started")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in bar:
        if not batch: continue

        images, targets = batch
        images = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.2f}")

    print(f"Epoch {epoch+1} Loss: {total_loss:.2f}")

torch.save(model.state_dict(), "/kaggle/working/maskrcnn_scratch.pth")

print("\n🎉 SCRATCH MODEL DONE")

🚀 Scratch Training on: cuda
📦 Scratch dataset size: 10000
🧠 Initializing SCRATCH model...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]



🚀 Scratch Training Started


Epoch 1/10: 100%|██████████| 2500/2500 [19:50<00:00,  2.10it/s, loss=0.31]


Epoch 1 Loss: 1374.90


...

Epoch 8 Loss: 557.64


Epoch 9/10: 100%|██████████| 2500/2500 [20:21<00:00,  2.05it/s, loss=0.21]


Epoch 9 Loss: 522.47


Epoch 10/10: 100%|██████████| 2500/2500 [20:15<00:00,  2.06it/s, loss=0.18]


Epoch 10 Loss: 490.58

🎉 SCRATCH MODEL DONE
